In [13]:
# INSTALACIÓN
!pip install pyamg pandas matplotlib -q

# LIBRERÍAS

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import gmres
import pyamg
import pandas as pd
import time

# CONTADOR DE ITERACIONES

class IterCounter:
    def __init__(self):
        self.niter = 0

    def __call__(self, rk=None):
        self.niter += 1

# FUNCIÓN DE EXPERIMENTO

def ejecutar_metodo(A, b, M=None):

    contador = IterCounter()

    inicio = time.time()

    x, info = gmres(
        A,
        b,
        M=M,
        callback=contador,
        rtol=1e-6,
        maxiter=5000
    )

    tiempo = time.time() - inicio

    residual = np.linalg.norm(b - A @ x) / np.linalg.norm(b)

    return tiempo, contador.niter, residual

# TAMAÑOS DE MATRIZ

tamanos = [1024, 4096, 16384]

resultados = []

# EXPERIMENTOS

for n in tamanos:

    print(f"\nProcesando matriz n={n} - Untitled-1.ipynb:55")

    diag = 2*np.ones(n)
    off = -1*np.ones(n-1)

    A = sp.diags(
        [off, diag, off],
        [-1, 0, 1],
        format='csr'
    )

    x_real = np.ones(n)

    b = A @ x_real

    # GMRES

    tiempo, iters, residual = ejecutar_metodo(A, b)

    resultados.append([
        n,
        "GMRES",
        tiempo,
        iters,
        residual
    ])

    # GMRES + AMG

    ml = pyamg.smoothed_aggregation_solver(A)

    M_amg = ml.aspreconditioner()

    tiempo, iters, residual = ejecutar_metodo(
        A,
        b,
        M=M_amg
    )

    resultados.append([
        n,
        "GMRES + AMG",
        tiempo,
        iters,
        residual
    ])

# TABLA FINAL

tabla = pd.DataFrame(
    resultados,
    columns=[
        "Tamaño",
        "Método",
        "Tiempo (s)",
        "Iteraciones",
        "Residual"
    ]
)

print("\nRESULTADOS FINALES\n - Untitled-1.ipynb:115")

display(tabla)


Procesando matriz n=1024 - Untitled-1.ipynb:55


C:\Users\danie\AppData\Local\Temp\ipykernel_21368\10201525.py:72: DeprecationWarning: scipy.sparse.linalg.gmres called without specifying `callback_type`. The default value will be changed in a future release. For compatibility, specify a value for `callback_type` explicitly, e.g., ``gmres(..., callback_type='pr_norm')``, or to retain the old behavior ``gmres(..., callback_type='legacy')``
  tiempo, iters, residual = ejecutar_metodo(A, b)
C:\Users\danie\AppData\Local\Temp\ipykernel_21368\10201525.py:88: DeprecationWarning: scipy.sparse.linalg.gmres called without specifying `callback_type`. The default value will be changed in a future release. For compatibility, specify a value for `callback_type` explicitly, e.g., ``gmres(..., callback_type='pr_norm')``, or to retain the old behavior ``gmres(..., callback_type='legacy')``
  tiempo, iters, residual = ejecutar_metodo(



Procesando matriz n=4096 - Untitled-1.ipynb:55

Procesando matriz n=16384 - Untitled-1.ipynb:55

RESULTADOS FINALES
 - Untitled-1.ipynb:115


,Tamaño,Método,Tiempo (s),Iteraciones,Residual
0,1024,GMRES,2.132152,5000,1.204397e-04
1,1024,GMRES + AMG,0.023700,5,3.127712e-07
2,4096,GMRES,2.141887,5000,9.492424e-05
3,4096,GMRES + AMG,0.022163,7,8.711113e-08
4,16384,GMRES,9.227276,5000,9.492424e-05
5,16384,GMRES + AMG,0.123179,6,5.491259e-08


In [14]:
!python examples/all_in_one.py \
--problem 1d_laplacian \
-n 1024 \
--preconditioners gmres gnp amg \
--epochs 20 \
--rtol 1e-6 \
--max_iters 1000

Input options. "n" and "location" may be adjusted according to "problem"
{
  "problem": "1d_laplacian",
  "n": 1024,
  "location": "~/data/SuiteSparse/ssget/mat",
  "rhs": "x_all_ones",
  "restart": 10,
  "max_iters": 1000,
  "timeout": null,
  "rtol": 1e-06,
  "preconditioners": [
    "gmres",
    "gnp",
    "amg"
  ],
  "inner_iters": 10,
  "inner_rtol": 1e-06,
  "training_data": "x_mix",
  "m": 40,
  "num_layers": 8,
  "embed": 16,
  "hidden": 32,
  "drop_rate": 0.0,
  "disable_scale_input": false,
  "precision": "float32",
  "batch_size": 16,
  "grad_accu_steps": 1,
  "epochs": 20,
  "lr": 0.001,
  "weight_decay": 0.0,
  "num_workers": 0,
  "model_file": null,
  "save_model": false,
  "ilu_factors_file": null,
  "save_ilu_factors": false,
  "fill_factor": null,
  "bjac_lu_factors_file": null,
  "save_bjac_lu_factors": false,
  "block_size": 32,
  "out_path": "./dump/",
  "out_file_prefix": null,
  "hide_solver_bar": false,
  "hide_training_bar": false
}

Matrix A: name = 1d_laplaci

C:\Users\danie\OneDrive\Documentos\GNP\GNP\problems.py:19: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  A = torch.sparse_csc_tensor(A.indptr, A.indices, A.data,
C:\Users\danie\OneDrive\Documentos\GNP\GNP\problems.py:19: UserWarning: Sparse CSC tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  A = torch.sparse_csc_tensor(A.indptr, A.indices, A.data,

Solve: 100%|██████

In [15]:
!pip install -e .


Obtaining file:///C:/Users/danie/OneDrive/Documentos/GNP
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for GNP (pyproject.toml): started
  Building editable for GNP (pyproject.toml): finished with status 'done'
  Created wheel for GNP: filename=gnp-1.0-0.editable-py3-none-any.whl size=6859 sha256=fae66e541b3518d4751aa997cb9f36567655157c8a2df132a205861b82adc0eb
  Stored in directory: C:\Users\danie\AppData\Local\Temp\pip-ephem-wheel-cache-56h32wyz\wheels\0e\94\8a\d76c9ba8938de67818c9050b0422d6703376e77086b93e76ee
Successfully

In [16]:
!python examples/all_in_one.py \
--problem 1d_laplacian \
-n 1024 \
--preconditioners gmres gnp amg \
--epochs 20 \
--rtol 1e-6 \
--max_iters 1000

Input options. "n" and "location" may be adjusted according to "problem"
{
  "problem": "1d_laplacian",
  "n": 1024,
  "location": "~/data/SuiteSparse/ssget/mat",
  "rhs": "x_all_ones",
  "restart": 10,
  "max_iters": 1000,
  "timeout": null,
  "rtol": 1e-06,
  "preconditioners": [
    "gmres",
    "gnp",
    "amg"
  ],
  "inner_iters": 10,
  "inner_rtol": 1e-06,
  "training_data": "x_mix",
  "m": 40,
  "num_layers": 8,
  "embed": 16,
  "hidden": 32,
  "drop_rate": 0.0,
  "disable_scale_input": false,
  "precision": "float32",
  "batch_size": 16,
  "grad_accu_steps": 1,
  "epochs": 20,
  "lr": 0.001,
  "weight_decay": 0.0,
  "num_workers": 0,
  "model_file": null,
  "save_model": false,
  "ilu_factors_file": null,
  "save_ilu_factors": false,
  "fill_factor": null,
  "bjac_lu_factors_file": null,
  "save_bjac_lu_factors": false,
  "block_size": 32,
  "out_path": "./dump/",
  "out_file_prefix": null,
  "hide_solver_bar": false,
  "hide_training_bar": false
}

Matrix A: name = 1d_laplaci

C:\Users\danie\OneDrive\Documentos\GNP\GNP\problems.py:19: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  A = torch.sparse_csc_tensor(A.indptr, A.indices, A.data,
C:\Users\danie\OneDrive\Documentos\GNP\GNP\problems.py:19: UserWarning: Sparse CSC tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  A = torch.sparse_csc_tensor(A.indptr, A.indices, A.data,

Solve: 100%|██████

In [17]:
!python examples/all_in_one.py \
--problem 1d_laplacian \
-n 4096 \
--preconditioners gmres gnp amg \
--epochs 20 \
--rtol 1e-6 \
--max_iters 1000

Input options. "n" and "location" may be adjusted according to "problem"
{
  "problem": "1d_laplacian",
  "n": 4096,
  "location": "~/data/SuiteSparse/ssget/mat",
  "rhs": "x_all_ones",
  "restart": 10,
  "max_iters": 1000,
  "timeout": null,
  "rtol": 1e-06,
  "preconditioners": [
    "gmres",
    "gnp",
    "amg"
  ],
  "inner_iters": 10,
  "inner_rtol": 1e-06,
  "training_data": "x_mix",
  "m": 40,
  "num_layers": 8,
  "embed": 16,
  "hidden": 32,
  "drop_rate": 0.0,
  "disable_scale_input": false,
  "precision": "float32",
  "batch_size": 16,
  "grad_accu_steps": 1,
  "epochs": 20,
  "lr": 0.001,
  "weight_decay": 0.0,
  "num_workers": 0,
  "model_file": null,
  "save_model": false,
  "ilu_factors_file": null,
  "save_ilu_factors": false,
  "fill_factor": null,
  "bjac_lu_factors_file": null,
  "save_bjac_lu_factors": false,
  "block_size": 32,
  "out_path": "./dump/",
  "out_file_prefix": null,
  "hide_solver_bar": false,
  "hide_training_bar": false
}

Matrix A: name = 1d_laplaci

C:\Users\danie\OneDrive\Documentos\GNP\GNP\problems.py:19: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  A = torch.sparse_csc_tensor(A.indptr, A.indices, A.data,
C:\Users\danie\OneDrive\Documentos\GNP\GNP\problems.py:19: UserWarning: Sparse CSC tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  A = torch.sparse_csc_tensor(A.indptr, A.indices, A.data,

Solve: 100%|██████

In [18]:
!python examples/all_in_one.py \
--problem 1d_laplacian \
-n 16384 \
--preconditioners gmres gnp amg \
--epochs 20 \
--rtol 1e-6 \
--max_iters 1000

Input options. "n" and "location" may be adjusted according to "problem"
{
  "problem": "1d_laplacian",
  "n": 16384,
  "location": "~/data/SuiteSparse/ssget/mat",
  "rhs": "x_all_ones",
  "restart": 10,
  "max_iters": 1000,
  "timeout": null,
  "rtol": 1e-06,
  "preconditioners": [
    "gmres",
    "gnp",
    "amg"
  ],
  "inner_iters": 10,
  "inner_rtol": 1e-06,
  "training_data": "x_mix",
  "m": 40,
  "num_layers": 8,
  "embed": 16,
  "hidden": 32,
  "drop_rate": 0.0,
  "disable_scale_input": false,
  "precision": "float32",
  "batch_size": 16,
  "grad_accu_steps": 1,
  "epochs": 20,
  "lr": 0.001,
  "weight_decay": 0.0,
  "num_workers": 0,
  "model_file": null,
  "save_model": false,
  "ilu_factors_file": null,
  "save_ilu_factors": false,
  "fill_factor": null,
  "bjac_lu_factors_file": null,
  "save_bjac_lu_factors": false,
  "block_size": 32,
  "out_path": "./dump/",
  "out_file_prefix": null,
  "hide_solver_bar": false,
  "hide_training_bar": false
}

Matrix A: name = 1d_laplac

C:\Users\danie\OneDrive\Documentos\GNP\GNP\problems.py:19: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  A = torch.sparse_csc_tensor(A.indptr, A.indices, A.data,
C:\Users\danie\OneDrive\Documentos\GNP\GNP\problems.py:19: UserWarning: Sparse CSC tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  A = torch.sparse_csc_tensor(A.indptr, A.indices, A.data,

Solve: 100%|██████

In [19]:
import pandas as pd

tabla = pd.DataFrame({
    "Tamaño": [
        1024, 1024, 1024,
        4096, 4096, 4096,
        16384, 16384, 16384
    ],
    "Método": [
        "GMRES",
        "GMRES + AMG",
        "GMRES + GNP",
        "GMRES",
        "GMRES + AMG",
        "GMRES + GNP",
        "GMRES",
        "GMRES + AMG",
        "GMRES + GNP"
    ],
    "Residual Final": [
        5.4369e-04,
        1.3410e-07,
        8.2160e-02,
        5.4369e-04,
        6.5822e-07,
        5.9118e-02,
        5.4369e-04,
        6.6308e-07,
        2.9443e-01
    ],
    "Tiempo (s)": [
        1,
        0.012,
        1.662,
        1,
        0.049,
        6.704,
        6,
        0.192,
        25.131
    ]
})

display(tabla)

,Tamaño,Método,Residual Final,Tiempo (s)
0,1024,GMRES,5.436900e-04,1.000
1,1024,GMRES + AMG,1.341000e-07,0.012
2,1024,GMRES + GNP,8.216000e-02,1.662
3,4096,GMRES,5.436900e-04,1.000
4,4096,GMRES + AMG,6.582200e-07,0.049
5,4096,GMRES + GNP,5.911800e-02,6.704
6,16384,GMRES,5.436900e-04,6.000
7,16384,GMRES + AMG,6.630800e-07,0.192
8,16384,GMRES + GNP,2.944300e-01,25.131


In [20]:
import time

# ----------------------------
# Entrenamiento
# ----------------------------
inicio_train = time.time()

# código que entrena GNP

fin_train = time.time()
